In [ ]:
from pathlib import Path
import re
import sys
import uuid
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

BASE_DIR = Path(r"u:/Users/hm616288/Downloads/BRS-Agent")
PROMPT_PATH = BASE_DIR / "BRS Generation Prompt.txt"
DEFAULT_TOM = BASE_DIR / "ImagineR3.7_SSJSFJ_J5and6_Oct Release_TOM_V0.9.xlsx"

HEADER_ROW = [
    "Ref #",
    "Requirement Category",
    "Requirement Title",
    "Requirement Description",
    "Acceptance Criteria",
    "Requirement SME",
    "MoSCoW",
    "System",
    "Reference to Other Related Documents If Applicable",
]


def load_prompt(path: Path) -> str:
    if not path.exists():
        raise FileNotFoundError(f"Prompt file not found: {path}")
    return path.read_text(encoding="utf-8", errors="replace")


def infer_journey_name(tom_path: Path) -> str:
    name = tom_path.stem.replace("_TOM", "")
    name = re.sub(r"_V\d+(?:\.\d+)?$", "", name)
    name = name.replace("_", " ")
    return name.strip()


def normalize_text(v):
    if v is None:
        return ""
    s = str(v).strip()
    s = re.sub(r"\s+", " ", s)
    return s


def extract_candidate_phrases(wb):
    phrases = []
    for ws in wb.worksheets:
        for row in ws.iter_rows(values_only=True):
            parts = [normalize_text(v) for v in row if normalize_text(v)]
            text = " ".join(parts)
            text = re.sub(r"\s+", " ", text).strip()
            if len(text) < 30:
                continue
            low = text.lower()
            if any(k in low for k in ["coverage", "quote", "order", "cart", "plp", "fulfilment", "verification", "journey", "submission", "validation"]):
                phrases.append((ws.title, text))
    return phrases


def determine_category(text):
    low = text.lower()
    if any(k in low for k in ["coverage", "onesq", "fwa", "override coverage"]):
        return "Coverage"
    if any(k in low for k in ["quote created", "quote status", "quote", "resume"]) and "coverage" not in low:
        return "Quote"
    if any(k in low for k in ["plp", "cart", "add plp", "page navigation"]) and "quote" not in low:
        return "Cart"
    if any(k in low for k in ["create order", "order type", "order submission", "confirmation", "submission", "fulfilment order"]):
        return "Order"
    if any(k in low for k in ["fulfilment", "post verification", "verification", "shipment", "appointment"]):
        return "Fulfilment"
    if any(k in low for k in ["journey", "validation", "screen", "page", "status"]):
        return "SFJ"
    return "Quote"


def make_title(text):
    cleaned = text.strip()
    cleaned = cleaned.replace("Validate that ", "")
    cleaned = cleaned.replace("Validate ", "")
    cleaned = re.sub(r"\s+", " ", cleaned)
    cleaned = cleaned.rstrip(".")
    if len(cleaned) > 110:
        cleaned = cleaned[:107].rstrip() + "..."
    return cleaned


def build_requirement_rows(wb):
    phrases = extract_candidate_phrases(wb)
    grouped = {}
    for _, text in phrases:
        cat = determine_category(text)
        grouped.setdefault(cat, []).append(text)

    requirements = []
    order = ["Coverage", "Quote", "Cart", "Order", "Fulfilment", "SFJ"]
    for index, cat in enumerate(order, 1):
        items = grouped.get(cat, [])
        if not items:
            continue
        dedup = []
        seen = set()
        for item in items:
            key = re.sub(r"\s+", " ", item).lower()
            if key not in seen:
                seen.add(key)
                dedup.append(item)

        for item in dedup[:2]:
            title = make_title(item)
            description = "The solution must " + title.lower().rstrip(".") + "."
            criteria_base = re.sub(r"\s+", " ", item).strip()
            criteria = "AC1: The system follows the TOM-defined behaviour described in the scenario: " + criteria_base[:250] + ".\nAC2: The system displays the relevant status or message and prevents the user from progressing when the condition is not satisfied."
            system = "Journey Platform"
            low = item.lower()
            if "onesq" in low:
                system = "OneSQ"
            elif "siebel" in low:
                system = "Siebel"
            elif "nbn" in low:
                system = "NBN"
            requirements.append({
                "ref": f"FR{index:03d}",
                "category": cat,
                "title": title,
                "description": description,
                "criteria": criteria,
                "sme": "",
                "moscow": "Must",
                "system": system,
                "ref_docs": "",
            })

    if not requirements:
        requirements.append({
            "ref": "FR001",
            "category": "Quote",
            "title": "Journey validation and system behaviour",
            "description": "The solution must apply the journey validation and business rules described in the attached TOM for the selected customer and journey stage.",
            "criteria": "AC1: The system evaluates the conditions defined in the TOM before proceeding to the next step.\nAC2: The system displays the associated status, message, or result for the action or validation outcome.",
            "sme": "",
            "moscow": "Must",
            "system": "Journey Platform",
            "ref_docs": "",
        })

    return requirements


def style_headers(ws):
    for cell in ws[4]:
        cell.font = Font(color="FFFFFF", bold=True)
        cell.fill = PatternFill("solid", fgColor="2F75B5")
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = Border(bottom=Side(style="thin", color="D9E2F3"))


def apply_table_style(ws):
    ws.freeze_panes = "A5"
    ws.auto_filter.ref = f"A4:I{ws.max_row}"
    for col_letter, width in {"A": 12, "B": 30, "C": 34, "D": 62, "E": 96, "F": 20, "G": 12, "H": 14, "I": 42}.items():
        ws.column_dimensions[col_letter].width = width
    for row in ws.iter_rows(min_row=5, min_col=1, max_col=9):
        for cell in row:
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.border = Border(
                left=Side(style="thin", color="D9E2F3"),
                right=Side(style="thin", color="D9E2F3"),
                top=Side(style="thin", color="D9E2F3"),
                bottom=Side(style="thin", color="D9E2F3"),
            )


def create_brs_workbook(journey_name: str, tom_path: Path, output_path: Path):
    title_text = f"Business Requirements Specification - {journey_name}"
    subtitle_text = "Functional requirements derived from TOM - Testcase Verification"
    title_style = Font(color="FFFFFF", bold=True, size=12)
    subtitle_style = Font(color="000000", bold=True, size=10)

    wb = Workbook()
    ws = wb.active
    ws.title = "4. Functional Requirements"
    ws.sheet_view.showGridLines = False

    ws.merge_cells("A1:I1")
    ws["A1"] = title_text
    ws["A1"].font = title_style
    ws["A1"].alignment = Alignment(horizontal="center", vertical="center")
    ws["A1"].fill = PatternFill("solid", fgColor="1F3B6D")

    ws.merge_cells("A2:I2")
    ws["A2"] = subtitle_text
    ws["A2"].font = subtitle_style
    ws["A2"].alignment = Alignment(horizontal="center", vertical="center")
    ws["A2"].fill = PatternFill("solid", fgColor="D9EAF7")

    ws.append(HEADER_ROW)
    style_headers(ws)

    tom_wb = load_workbook(tom_path, data_only=True)
    for req in build_requirement_rows(tom_wb):
        ws.append([
            req["ref"],
            req["category"],
            req["title"],
            req["description"],
            req["criteria"],
            req["sme"],
            req["moscow"],
            req["system"],
            req["ref_docs"],
        ])

    apply_table_style(ws)

    for sheet_name in ["1. Scope", "Image", "Ref"]:
        new_ws = wb.create_sheet(sheet_name)
        new_ws["A1"] = "Required section placeholder"
        new_ws["A1"].font = Font(bold=True)
        if sheet_name == "Ref":
            new_ws["A1"] = "Reference"

    try:
        wb.save(output_path)
        return output_path
    except PermissionError:
        fallback = output_path.with_name(f"{output_path.stem}_{uuid.uuid4().hex[:8]}.xlsx")
        wb.save(fallback)
        return fallback


tom_path = DEFAULT_TOM
prompt_text = load_prompt(PROMPT_PATH)
journey_name = infer_journey_name(tom_path)
output_path = BASE_DIR / f"BRS_Generated_{journey_name}.xlsx"
create_brs_workbook(journey_name, tom_path, output_path)
print(f"BRS output created: {output_path}")
print(f"Prompt length: {len(prompt_text)}")
